# XLSR-53 against wav2vec2-base: does a bigger, multilingual encoder help?

**What this answers.** A frozen-feature probe on the laptop showed XLSR-53's
representations are much better than wav2vec2-base's on this task — on the **unseen
Tortoise tool**, 38.20% against 51.77% EER, a gap whose whole 95% interval is below
zero (`docs/results/encoder_ablation_v1.md`). This notebook finds out whether that
survives **training**.

**Why it needs Kaggle.** On the laptop, XLSR would not learn under the project's LoRA
recipe (loss stuck at 0.693, a coin toss), and the full S3 fine-tune needs 6.45 GB on
a 6 GB card. A T4 has 16 GB.

**Three stages, each one gated on the last:**

| Stage | What | Time (T4) |
|---|---|---|
| 1 | Learning-rate search: does XLSR + LoRA learn *at all*? | ~20 min |
| 2 | Full XLSR + LoRA against wav2vec2-base + LoRA, same recipe, scored on 3 test sets | ~50 min |
| 3 | S3 rebuilt on XLSR — full fine-tune, scored on the same 3 sets | ~90 min |

Stage 2 always runs: at the best rate stage 1 finds, or, if none learns within the
short search, at 1e-4 (wav2vec2-base's rate) for the full 5 epochs, so the comparison
is settled by a full run and not by a 300-step trial.

A model whose output goes constant ("collapsed") is recorded as a result. If it
collapses after a healthy epoch, its best checkpoint from before the collapse is
scored.

**Nothing stops the run part-way.** A step that fails is recorded loudly (it heads
the verdict as `RUN HEALTH` and its error text goes to `FAILURES.txt`) and the
remaining stages still run, so every result that can be produced is produced and
saved. The full output of every command is kept in `run_log.txt`.

`xlsr_results.zip` is refreshed after every stage, so a failure or timeout late in
the run never loses the stages that finished.

The 10-minute test run (`SMOKE = True`) has already passed on Kaggle, so `SMOKE` is
set to `False`: commit this as it is for the real numbers.

**Inputs to attach:** `saikrishnareddy9/codemix-bundle-normalised`.
**Settings:** GPU T4 x1, Internet **On** (XLSR-53 downloads from Hugging Face).

## 1. Settings — the only cell you edit

In [ ]:
# ==================================================================================
#  THE ONLY CELL YOU EDIT
# ==================================================================================
SMOKE = False              # False -> the real experiment, ~2-3 hours.
                           # True  -> ~10 min check that every step works (already
                           #          passed on Kaggle); its numbers mean NOTHING.

RUN_STAGE_3 = True         # S3 rebuilt on XLSR (full fine-tune), ~1.5 h. False skips it.
SAVE_CHECKPOINTS = True    # copy the trained XLSR models to the output (~1.2 GB each)

# ---- nothing below this line should need changing --------------------------------
REPO_HOST = "github.com/Mounika-Reddy-0802/codemix-deepfake-detection.git"
BRANCH = "main"

LR_GRID = [1e-4, 3e-4, 1e-3, 3e-3]   # stage 1 tries each of these for XLSR + LoRA
SEARCH_STEPS = 300                   # training steps per learning-rate trial
CHANCE = 0.693                       # ln 2: the loss of a coin toss on this task
LEARNS_BELOW = 0.60                  # a trial "learns" if its last loss falls below this
W2V2_LR = 1e-4                       # wav2vec2-base's known-good rate (the deployed S2's)
S3_LR = 1e-5                         # the S3 recipe's rate for a full fine-tune

TRAIN_MANIFEST = "data/manifests/codemix_adapt_train_rvc_channel20.csv"
DEV_MANIFEST = "data/manifests/codemix_adapt_dev_channel20.csv"
EVALS = {
    "eval_pool": "data/manifests/codemix_eval_channel20.csv",       # seen tool (XTTS)
    "cm04":      "data/manifests/score_cm04_norm_channel20.csv",    # UNSEEN tool (Tortoise)
    "rvc_test":  "data/manifests/rvc_holdout_test_channel20.csv",   # RVC, unseen speakers
}

# Shortcut floors (channel condition) and the published results to compare against.
FLOOR = {"eval_pool": 10.01, "cm04": 25.79, "rvc_test": 22.46}
PUBLISHED = {
    "S2: LoRA from English S1, wav2vec2-base (deployed)": {"eval_pool": 2.75, "cm04": 30.99, "rvc_test": 5.01},
    "S3: native, wav2vec2-base":                          {"eval_pool": 1.79, "cm04": 31.67, "rvc_test": 5.98},
}
SCORE_ARGS = "--device cuda --batch-size 32 --num-workers 4 --max-seconds 4.0"

## 2. Setup: clone the repository, install, find the data

In [ ]:
import glob, json, os, re, shutil, subprocess, sys, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "codemix-deepfake-detection"
OUT = WORK / "xlsr_results"


def sh(cmd, check=True):
    """Run a short command and show its output."""
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True,
                       cwd=REPO if REPO.exists() else None)
    if r.stdout.strip():
        print(r.stdout[-2500:])
    if r.returncode and r.stderr.strip():
        print(r.stderr[-2500:])
    if check and r.returncode:
        raise SystemExit(f"failed ({r.returncode}): {cmd}")
    return r


def run_logged(args):
    """Run a long command, streaming its output live, and return all of it.

    ``args`` is a list, run without a shell, so no path can be mangled by quoting.
    """
    args = [str(a) for a in args]
    print("$", " ".join(args), flush=True)
    proc = subprocess.Popen(args, text=True, errors="replace", cwd=REPO,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = []
    with open(OUT / "run_log.txt", "a", encoding="utf-8") as log:  # the full output is kept
        log.write(f"\n$ {' '.join(args)}\n")
        for line in proc.stdout:
            lines.append(line)
            log.write(line)
            if not any(noise in line for noise in ("Warning", "warn(", "Some weights", "You should")):
                print(line, end="", flush=True)
    proc.wait()
    return proc.returncode, "".join(lines)


def _make_writable(func, path, _):
    """git marks its pack files read-only; clear that so a re-run can remove them."""
    os.chmod(path, 0o700)
    func(path)


for attempt in range(1, 4):  # GitHub occasionally drops a connection; try three times
    if REPO.exists():  # always start from a clean clone
        shutil.rmtree(REPO, onerror=_make_writable)
    if subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1",
                       f"https://{REPO_HOST}", str(REPO)]).returncode == 0:
        break
    print(f"clone attempt {attempt} failed; retrying in 20 s", flush=True)
    time.sleep(20)
else:
    raise SystemExit("Could not clone the repository after 3 attempts. Is Internet switched On?")
os.chdir(REPO)
sys.path.insert(0, str(REPO))
# results from an earlier run in this session must not mix into this one
shutil.rmtree(OUT, ignore_errors=True)
(WORK / "xlsr_results.zip").unlink(missing_ok=True)
OUT.mkdir()


def save_progress():
    """Zip the results so far (not the models) into /kaggle/working/xlsr_results.zip.

    Called after every stage, so whatever has finished is in the output even if a
    later stage fails or the session runs out of time.
    """
    import zipfile
    staging = WORK / "xlsr_results.zip.tmp"
    with zipfile.ZipFile(staging, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(OUT.rglob("*")):
            if f.is_file() and f.suffix != ".pt":
                zf.write(f, f.relative_to(WORK))
    staging.replace(WORK / "xlsr_results.zip")


FAILURES = []  # every step that failed; the verdict lists them first


def tail(text, n=30):
    """The last n non-empty lines of a command's output: where its error is."""
    return "\n".join([line for line in text.splitlines() if line.strip()][-n:])


def failed(where, detail):
    """Record a failed step loudly, save it, and let the run continue.

    The steps that do work still finish and everything is saved, so one failure never
    costs the whole run. Each failure goes to FAILURES.txt and heads the verdict.
    """
    FAILURES.append({"where": where, "detail": detail})
    print(f"\n!!!!!! FAILED: {where}\n{detail}\n!!!!!! recorded; the run continues\n", flush=True)
    try:
        (OUT / "FAILURES.txt").write_text(
            "\n\n".join(f"== {f['where']}\n{f['detail']}" for f in FAILURES), encoding="utf-8")
        save_progress()
    except Exception as e:  # recording must never be what stops the run
        print("(could not write FAILURES.txt:", e, ")")


COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, text=True,
                        capture_output=True).stdout.strip()
sh("git log --oneline -1")
sh(f"{sys.executable} -m pip install -q 'transformers>=4.57,<5' soundfile==0.12.1 librosa==0.11.0",
   check=False)

import pandas as pd
from src.data.scoring_manifests import missing_files

channel = [p for p in glob.glob("/kaggle/input/**/channel/clips", recursive=True) if os.listdir(p)]
if not channel:
    raise SystemExit("Attach the dataset saikrishnareddy9/codemix-bundle-normalised (Add Input).")
ROOT = str(Path(channel[0]).parent)
print("data root:", ROOT)

for name, manifest in {"train": TRAIN_MANIFEST, "dev": DEV_MANIFEST, **EVALS}.items():
    gone = missing_files(pd.read_csv(manifest), ROOT)
    print(f"  {name:10s} {manifest:58s} missing {len(gone)}")
    assert not gone, f"{manifest} has clips not in the dataset: {gone[:3]}"
print("every clip every stage needs is present")

## 3. Write the training configs, then check them against the dataset rule

The configs are written here rather than read from the repository, so this notebook
does not depend on any branch being merged. `config_guard` then opens each one's real
manifests and refuses Tortoise, evaluation-only corpora, held-out speakers, or an S3
run that starts from the English checkpoint.

In [ ]:
import yaml

CONFIGS = REPO / "configs"


def make_config(name, encoder, lr, epochs, *, s3=False, max_steps=None, dev_subset=None):
    cfg = {
        "encoder": encoder,
        "train_manifest": TRAIN_MANIFEST,
        "dev_manifest": DEV_MANIFEST,
        "out_dir": f"checkpoints/w12/{name}",
        "lr": lr,
        "weight_decay": 1.0e-5,
        "batch_size": 8,
        "max_seconds": 4.0,
        "epochs": epochs,
        "num_workers": 4,
        "seed": 1234,
        "device": "cuda",
        "amp": True,
        "class_weighted": True,
        "log_every": 25,
        "wandb_mode": "disabled",
    }
    if s3:  # full fine-tune from the pretrained encoder, exactly the S3 recipe
        cfg.update({"system": "s3_native", "freeze_feature_extractor": True, "freeze_encoder": False})
    else:   # LoRA adaptation from the pretrained encoder
        cfg.update({"lora": True, "lora_r": 8, "lora_alpha": 16.0, "lora_dropout": 0.05})
    if max_steps:
        cfg["max_steps"] = max_steps
    if dev_subset:
        cfg["dev_subset"] = dev_subset
    path = CONFIGS / f"w12_{name}.yaml"
    path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    return str(path.relative_to(REPO))


def lr_tag(lr):
    return f"{lr:.0e}".replace("-0", "-")


search_cfgs = {lr: make_config(f"search_xlsr_lr{lr_tag(lr)}", "xlsr-53", lr, 1,
                               max_steps=SEARCH_STEPS, dev_subset=200) for lr in LR_GRID}
w2v2_cfg = make_config("w2v2base_lora", "wav2vec2-base", W2V2_LR, 5)
s3_cfg = make_config("s3_native_xlsr", "xlsr-53", S3_LR, 10, s3=True)

all_cfgs = list(search_cfgs.values()) + [w2v2_cfg, s3_cfg]
sh(f"{sys.executable} -m pytest tests/test_splits.py tests/test_config_guard.py -q")
sh(f"{sys.executable} -m src.training.config_guard " + " ".join(all_cfgs))

## 4. Stage 1 — does XLSR + LoRA learn at all?

The laptop run sat at 0.693 — the loss of a coin toss — for a whole epoch at the
project's usual learning rate. This tries four rates for a short run each and keeps
the one whose loss falls furthest. A rate whose output goes constant ("collapsed")
is recorded and the search moves on. **If none falls below 0.60 within 300 steps,
stage 2 still runs XLSR for the full 5 epochs at 1e-4**, wav2vec2-base's rate, so a
short trial never decides the comparison on its own.

In [ ]:
search = {}
BEST_LR = None
try:
    LOSS_LINE = re.compile(r"step \d+/\d+\s+loss ([0-9.]+|nan|inf)")
    DEV_LINE = re.compile(r"dev EER ([0-9.]+)\s+score std ([0-9.]+)")

    search = {}
    for lr, cfg in search_cfgs.items():
        t0 = time.time()
        code_, text = run_logged([sys.executable, "-m", "src.training.train", "--config", cfg,
                                  "--data-root", ROOT, "--device", "cuda"] + (["--smoke"] if SMOKE else []))
        # train.py's collapse guard stops a model whose scores no longer vary. For a
        # learning-rate search that is a RESULT -- "does not learn at this rate" -- so the
        # search records it and moves on. Anything else that stops training is a real error.
        collapsed = "model collapsed" in text
        crashed = code_ != 0 and not collapsed
        if crashed:
            failed(f"stage 1 trial at learning rate {lr:.0e} (exit {code_})", tail(text))
        losses = [float(x) for x in LOSS_LINE.findall(text)]
        dev = DEV_LINE.findall(text)
        search[lr] = {
            "ok": code_ == 0,
            "collapsed": collapsed,
            "crashed": crashed,
            "first_loss": losses[0] if losses else None,
            "last_loss": losses[-1] if losses else None,
            "dev_eer": float(dev[-1][0]) if dev else None,
            "score_std": float(dev[-1][1]) if dev else None,
            "minutes": round((time.time() - t0) / 60, 1),
        }
        # the trial's checkpoints are not needed and XLSR's are ~2.4 GB each
        shutil.rmtree(REPO / "checkpoints" / "w12" / f"search_xlsr_lr{lr_tag(lr)}", ignore_errors=True)

    print("\nSTAGE 1 — XLSR + LoRA learning-rate search (chance loss = 0.693)\n")
    print(f"{'learning rate':>14}{'first loss':>12}{'last loss':>12}{'dev EER':>10}{'score std':>11}  verdict")
    for lr, r in search.items():
        learns = not (r["collapsed"] or r["crashed"]) and r["last_loss"] is not None and r["last_loss"] < LEARNS_BELOW
        fmt = lambda v, f: "-" if v is None else format(v, f)
        print(f"{lr:>14.0e}{fmt(r['first_loss'], '.3f'):>12}{fmt(r['last_loss'], '.3f'):>12}"
              f"{fmt(r['dev_eer'], '.3f'):>10}{fmt(r['score_std'], '.4f'):>11}  "
              f"{'LEARNS' if learns else 'CRASHED (see FAILURES)' if r['crashed'] else 'collapsed: constant output' if r['collapsed'] else 'stuck near chance'}")

    learning = {lr: r for lr, r in search.items()
                if not (r["collapsed"] or r["crashed"]) and r["last_loss"] is not None and r["last_loss"] < LEARNS_BELOW}
    if SMOKE:
        BEST_LR = LR_GRID[0]
        print("\nSMOKE: 20 steps cannot show learning; continuing with", BEST_LR, "to test the chain.")
    elif learning:
        BEST_LR = min(learning, key=lambda lr: learning[lr]["last_loss"])
        print(f"\nXLSR learns. Best learning rate: {BEST_LR:.0e}. Stage 2 will use it.")
    else:
        BEST_LR = None
        print(f"\nXLSR did not learn within {SEARCH_STEPS} steps at any of these rates.")
        print(f"Stage 2 still runs XLSR for the full 5 epochs at {W2V2_LR:.0e} (wav2vec2-base's rate),")
        print("so a short trial does not decide the comparison on its own.")
    (OUT / "stage1_lr_search.json").write_text(json.dumps({str(k): v for k, v in search.items()}, indent=2))
    save_progress()
except Exception:
    failed('stage 1' + ": unexpected error", traceback.format_exc())

## 5. Stage 2 — XLSR + LoRA against wav2vec2-base + LoRA, same recipe

Both start from the pretrained encoder with no English stage, read the same
manifests, and are scored on the same three test sets. The only difference is the
encoder (and XLSR's learning rate from stage 1).

In [ ]:
results = {}
COLLAPSED = []
UNSCORED = []
XLSR_LR = W2V2_LR
STAGE2_DONE = False
try:
    results = {}
    COLLAPSED = []  # models whose training stopped because their output went constant
    UNSCORED = []   # ...of those, the ones that collapsed before any checkpoint was saved


    def score(tag, ckpt, manifest, encoder):
        args = [sys.executable, "-m", "src.training.evaluate", "--checkpoint", ckpt, "--manifest", manifest,
                "--encoder", encoder, "--data-root", ROOT, *SCORE_ARGS.split(),
                "--scores-out", OUT / f"{tag}_scores.csv", "--out", OUT / f"{tag}.json"]
        if SMOKE:
            args += ["--limit", "200"]
        code_, text = run_logged(args)
        try:
            pooled = json.loads((OUT / f"{tag}.json").read_text())["pooled"] if code_ == 0 else None
        except Exception:
            pooled = None
        if not pooled or "eer" not in pooled:
            failed(f"scoring {tag} (exit {code_})", tail(text))
            return False
        results[tag] = pooled
        print(f"  {tag:40s} EER {pooled['eer'] * 100:6.2f}%")
        return True


    def train_and_score(name, cfg, encoder):
        t0 = time.time()
        code_, text = run_logged([sys.executable, "-m", "src.training.train", "--config", cfg,
                                  "--data-root", ROOT, "--device", "cuda"] + (["--smoke"] if SMOKE else []))
        ckpt = REPO / "checkpoints" / "w12" / name / "best.pt"
        if code_ and "model collapsed" in text:
            COLLAPSED.append(name)
            if not ckpt.is_file():
                print(f"!! {name}: COLLAPSED in its first epoch -- its output went constant, so it "
                      "did not learn. Recorded as a result; there is no checkpoint to score.")
                UNSCORED.append(name)
                return False
            print(f"!! {name}: COLLAPSED in a later epoch. Scoring its best checkpoint, saved "
                  "before the collapse.")
        elif code_:
            failed(f"training {name} (exit {code_})", tail(text))
            return False
        scored = [score(f"{name}__{ev}", ckpt, manifest, encoder) for ev, manifest in EVALS.items()]
        print(f"[{name}] trained and scored in {(time.time() - t0) / 60:.1f} min")
        save_progress()
        (REPO / "checkpoints" / "w12" / name / "last.pt").unlink(missing_ok=True)  # disk
        return all(scored)


    XLSR_LR = BEST_LR if BEST_LR is not None else W2V2_LR
    print(f"Stage 2: wav2vec2-base at {W2V2_LR:.0e}, XLSR-53 at {XLSR_LR:.0e}"
          + ("" if BEST_LR is not None else " (no rate learned in the short search)"))
    xlsr_cfg = make_config("xlsr53_lora", "xlsr-53", XLSR_LR, 5)
    guard = sh(f"{sys.executable} -m src.training.config_guard {xlsr_cfg}", check=False)
    ok_base = train_and_score("w2v2base_lora", w2v2_cfg, "wav2vec2-base")
    if guard.returncode:  # the dataset rule refused this config: it must not be trained
        failed("config_guard refused the XLSR stage-2 config", (guard.stdout + guard.stderr)[-2500:])
        ok_xlsr = False
    else:
        ok_xlsr = train_and_score("xlsr53_lora", xlsr_cfg, "xlsr-53")
    STAGE2_DONE = ok_base and ok_xlsr
except Exception:
    failed('stage 2' + ": unexpected error", traceback.format_exc())

## 6. Stage 3 — S3 rebuilt on XLSR (full fine-tune)

What was originally asked for: the S3 recipe — every transformer weight trains, from
the pretrained encoder, on code-mixed audio only — with XLSR-53 in place of
wav2vec2-base. Compared against the published S3 on wav2vec2-base.

In [ ]:
STAGE3_DONE = False
try:
    if RUN_STAGE_3:
        STAGE3_DONE = train_and_score("s3_native_xlsr", s3_cfg, "xlsr-53")
    else:
        print("Stage 3 skipped (RUN_STAGE_3 = False).")
except Exception:
    failed('stage 3' + ": unexpected error", traceback.format_exc())

## 7. Results, verdict, and what to download

In [ ]:
verdict = []
try:
    def eer_of(tag):
        r = results.get(tag)
        return None if r is None else (100 * r["eer"], 100 * r.get("eer_ci_low", float("nan")),
                                       100 * r.get("eer_ci_high", float("nan")))


    rows = [("shortcut floor (channel)", {k: (v, None, None) for k, v in FLOOR.items()})]
    rows += [(label, {k: (v, None, None) for k, v in row.items()}) for label, row in PUBLISHED.items()]
    for name, label in [("w2v2base_lora", "NEW wav2vec2-base + LoRA (from pretrained)"),
                        ("xlsr53_lora", "NEW XLSR-53 + LoRA (from pretrained)"),
                        ("s3_native_xlsr", "NEW S3 native on XLSR-53")]:
        cells = {ev: eer_of(f"{name}__{ev}") for ev in EVALS}
        if all(c is not None for c in cells.values()):
            rows.append((label, cells))

    print("EER % on the channel condition (lower is better).  * = below its shortcut floor")
    print("[95% CI] where measured here.\n")
    print(f"{'system':<52}" + "".join(f"{ev:>24}" for ev in EVALS))
    for label, cells in rows:
        out = []
        for ev in EVALS:
            e, lo, hi = cells[ev]
            star = "*" if e < FLOOR[ev] and "floor" not in label else " "
            ci = f" [{lo:5.2f},{hi:5.2f}]" if lo is not None and lo == lo else ""
            out.append(f"{e:7.2f}{ci}{star}")
        print(f"{label:<52}" + "".join(f"{c:>24}" for c in out))

    verdict = []
    if SMOKE:
        expected = ["w2v2base_lora", "xlsr53_lora"] + (["s3_native_xlsr"] if RUN_STAGE_3 else [])
        scored = {tag.split("__")[0] for tag in results}
        missing = [name for name in expected if name not in scored and name not in UNSCORED]
        if missing:
            failed("smoke check", f"these models were never trained and scored: {missing}")
        elif not FAILURES:
            verdict.append(f"SMOKE RUN: all {len(expected)} models trained and scored on all 3 test sets. "
                           "Every step works. The numbers mean NOTHING -- set SMOKE = False and run again.")
    else:
        learned = [f"{lr:.0e}" for lr, r in search.items()
                   if not (r["collapsed"] or r.get("crashed")) and r["last_loss"] is not None and r["last_loss"] < LEARNS_BELOW]
        verdict.append(f"STAGE 1: within {SEARCH_STEPS} steps XLSR + LoRA learned at "
                       + (", ".join(learned) if learned else "NO rate tried")
                       + f"; stage 2 used {XLSR_LR:.0e}.")
        for name in COLLAPSED:
            verdict.append(f"NOTE: {name} collapsed (constant output) during training; "
                           + ("it could not be scored." if name in UNSCORED
                              else "its best checkpoint from before the collapse was scored."))
        if STAGE2_DONE:
            base, xl = eer_of("w2v2base_lora__cm04"), eer_of("xlsr53_lora__cm04")
            if xl[2] < base[0]:
                verdict.append(f"STAGE 2: XLSR clearly beats wav2vec2-base on the UNSEEN tool "
                               f"({xl[0]:.2f}% vs {base[0]:.2f}%; XLSR's whole interval is lower).")
            elif xl[1] > base[0]:
                verdict.append(f"STAGE 2: XLSR is WORSE than wav2vec2-base on the unseen tool "
                               f"({xl[0]:.2f}% vs {base[0]:.2f}%).")
            else:
                verdict.append(f"STAGE 2: no clear difference on the unseen tool "
                               f"({xl[0]:.2f}% vs {base[0]:.2f}%; the intervals overlap).")
            seen = eer_of("xlsr53_lora__eval_pool")[0]
            if seen > 40:
                verdict.append(f"        XLSR + LoRA stayed near chance even on the SEEN tool ({seen:.2f}%): "
                               "under LoRA it did not learn this task at all.")
        else:
            gone = [n for n in UNSCORED if n != "s3_native_xlsr"]
            verdict.append("STAGE 2: no comparison -- " + (f"collapsed in the first epoch: {gone}" if gone
                                                           else "a step failed (see RUN HEALTH)"))
        if "s3_native_xlsr" in UNSCORED:
            verdict.append(f"STAGE 3: S3 on XLSR COLLAPSED in its first epoch at learning rate {S3_LR:.0e} "
                           "-- its output went constant. Under the S3 recipe, XLSR-53 did not learn this task.")
        if RUN_STAGE_3 and not STAGE3_DONE and "s3_native_xlsr" not in UNSCORED:
            verdict.append("STAGE 3: did not complete -- a step failed (see RUN HEALTH)")
        if STAGE3_DONE:
            s3 = eer_of("s3_native_xlsr__cm04")
            old = PUBLISHED["S3: native, wav2vec2-base"]["cm04"]
            verdict.append(f"STAGE 3: S3 on XLSR reaches {s3[0]:.2f}% on the unseen tool, against "
                           f"{old:.2f}% for S3 on wav2vec2-base.")
            if s3[2] < FLOOR["cm04"]:
                verdict.append("        It CLEARS the unseen-tool floor (25.79%) — a headline result.")
            else:
                verdict.append("        It does not clear the unseen-tool floor (25.79%).")
    verdict.insert(0, f"RUN HEALTH: {len(FAILURES)} step(s) FAILED -- each one is in FAILURES.txt; the "
                      "results here cover only the steps that ran." if FAILURES
                      else "RUN HEALTH: every step ran without error.")
    print("\nVERDICT")
    for line in verdict:
        print(" ", line)
except Exception:
    failed('verdict' + ": unexpected error", traceback.format_exc())

In [ ]:
# Save the summary and the trained models. Runs even if the verdict above failed.

try:
    (OUT / "summary.json").write_text(json.dumps({
        "mode": "smoke" if SMOKE else "full",
        "branch": BRANCH,
        "commit": COMMIT,
        "stage1_best_lr": BEST_LR,
        "stage2_xlsr_lr": XLSR_LR,
        "stage1": {str(k): v for k, v in search.items()},
        "results": results,
        "floors": FLOOR,
        "published": PUBLISHED,
        "verdict": verdict,
        "collapsed": COLLAPSED,
        "collapsed_unscored": UNSCORED,
        "failures": FAILURES,
    }, indent=2, default=str))
    if SAVE_CHECKPOINTS and not SMOKE:
        for name in ("xlsr53_lora", "s3_native_xlsr"):
            src = REPO / "checkpoints" / "w12" / name / "best.pt"
            if src.is_file():  # moved, not copied: everything in /kaggle/working is saved
                shutil.move(str(src), str(OUT / f"{name}_best.pt"))
except Exception:
    failed('saving the summary' + ": unexpected error", traceback.format_exc())
save_progress()
print("RUN HEALTH:", "every step ran without error." if not FAILURES
      else f"{len(FAILURES)} step(s) FAILED -- details in FAILURES.txt:")
for f in FAILURES:
    print("   FAILED:", f["where"])
print("\nEverything is in xlsr_results/ and xlsr_results.zip (Output tab).")

## 8. Download everything as one zip

Creates `xlsr_results.zip` (the summary and every per-clip score file) and shows
download links. The trained models are linked separately because they are large.

In [ ]:
# ===== DOWNLOAD THE RESULTS =====
import os
import zipfile
from pathlib import Path

from IPython.display import FileLink, display

WORKING = Path("/kaggle/working")
RESULTS = WORKING / "xlsr_results"
ZIP = WORKING / "xlsr_results.zip"

os.chdir(WORKING)  # download links are relative to /kaggle/working

if not RESULTS.is_dir() or not any(RESULTS.iterdir()):
    print("No results found in /kaggle/working/xlsr_results.")
    print("The experiment cells above must finish first (or the session restarted).")
else:
    with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(RESULTS.rglob("*")):
            if f.is_file() and f.suffix != ".pt":  # models are 1.2 GB each; linked below
                zf.write(f, f.relative_to(WORKING))
    with zipfile.ZipFile(ZIP) as zf:
        count, broken = len(zf.namelist()), zf.testzip()
    if broken:
        raise SystemExit(f"zip is damaged at {broken} -- run this cell again")
    print(f"Created xlsr_results.zip: {count} files, {ZIP.stat().st_size / 1e6:.1f} MB, checked OK")
    print("\nCLICK THIS LINK TO DOWNLOAD:")
    display(FileLink("xlsr_results.zip"))

    models = sorted(RESULTS.glob("*.pt"))
    if models:
        print("\nTrained models (about 1.2 GB each -- only if you need them):")
        for m in models:
            display(FileLink(f"xlsr_results/{m.name}"))

if globals().get("FAILURES"):
    print(f"\n!!! {len(FAILURES)} step(s) FAILED in this run -- see FAILURES.txt in the zip "
          "and RUN HEALTH in the verdict above.")